In [1]:
import os, random, numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from itertools import product
from pycocotools.coco import COCO


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "/kaggle/input/coco-2017-dataset/coco2017"
img_dir = os.path.join(data_dir, "train2017")
ann_file = os.path.join(data_dir, "annotations", "instances_train2017.json")


In [3]:
coco = COCO(ann_file)
img_ids = coco.getImgIds()
sampled_ids = random.sample(img_ids, 100)
valid_ids = []
for img_id in sampled_ids:
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)
    mask = np.zeros((128, 128), dtype=np.uint8)
    for ann in anns:
        rle = coco.annToMask(ann)
        rle = cv2.resize(rle, (128, 128), interpolation=cv2.INTER_NEAREST)
        mask = np.maximum(mask, rle)
    if mask.sum() > 0:
        valid_ids.append(img_id)
small_img_ids = random.sample(valid_ids, min(5, len(valid_ids)))


loading annotations into memory...
Done (t=21.83s)
creating index...
index created!


In [4]:
class CocoSegDataset(Dataset):
    def __init__(self, coco, img_dir, img_ids, target_size=(128,128)):
        self.coco = coco
        self.img_dir = img_dir
        self.img_ids = img_ids
        self.target_size = target_size
        self.transform = T.ToTensor()

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, img_info['file_name'])

        image = Image.open(img_path).convert("RGB").resize(self.target_size)
        mask = np.zeros(self.target_size, dtype=np.uint8)
        for ann in anns:
            rle = self.coco.annToMask(ann)
            rle = cv2.resize(rle, self.target_size)
            mask = np.maximum(mask, rle)

        image = self.transform(image)
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        return image, mask

dataset = CocoSegDataset(coco, img_dir, small_img_ids)


In [5]:
class UNet(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout_rate = dropout
        self.enc1 = self.conv_block(3, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = self.conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = self.conv_block(128, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = self.conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = self.conv_block(128, 64)
        self.out = nn.Conv2d(64, 1, kernel_size=1)

    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(self.dropout_rate)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        b = self.bottleneck(p2)
        u2 = self.up2(b)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        return self.out(d1)

#  Metrics
def iou(pred, target, threshold=0.5):
    pred_bin = (pred > threshold).float()
    intersection = (pred_bin * target).sum()
    union = pred_bin.sum() + target.sum() - intersection
    return (intersection + 1e-6) / (union + 1e-6)

def dice(pred, target, threshold=0.5):
    pred_bin = (pred > threshold).float()
    intersection = (pred_bin * target).sum()
    return (2 * intersection + 1e-6) / (pred_bin.sum() + target.sum() + 1e-6)

In [9]:
def init_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
        nn.init.kaiming_normal_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

def train_and_evaluate(dataset, lr, optimizer_name='Adam', dropout=0.1, batch_size=1):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model = UNet(dropout=dropout).to(device)
    model.apply(init_weights)
    optimizer = {
        'Adam': optim.Adam(model.parameters(), lr=lr),
        'RMSprop': optim.RMSprop(model.parameters(), lr=lr),
        'SGD': optim.SGD(model.parameters(), lr=lr)
    }[optimizer_name]
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(5):
        model.train()
        for img, mask in loader:
            img, mask = img.to(device), mask.to(device)
            optimizer.zero_grad()
            output = model(img)
            loss = criterion(output, mask)
            loss.backward()
            optimizer.step()

    model.eval()
    iou_scores, dice_scores = [], []
    with torch.no_grad():
        for img, mask in loader:
            img, mask = img.to(device), mask.to(device)
            output = torch.sigmoid(model(img))
            iou_scores.append(iou(output, mask))
            dice_scores.append(dice(output, mask))
    return np.mean(iou_scores), np.mean(dice_scores)


In [14]:
print("\n Manual Search")
manual_lrs = [0.001, 0.0005, 0.002]
manual_results = {}
for lr in manual_lrs:
    iou_val, dice_val = train_and_evaluate(dataset, lr=lr)
    manual_results[lr] = (iou_val, dice_val)
    print(f"LR={lr}: IoU={iou_val:.4f}, Dice={dice_val:.4f}")

# Grid Search
print("\n Grid Search")
grid_lrs = [0.001, 0.0005]
grid_bs = [1, 2]
grid_results = {}
for lr, bs in product(grid_lrs, grid_bs):
    iou_val, dice_val = train_and_evaluate(dataset, lr=lr, batch_size=bs)
    grid_results[(lr, bs)] = (iou_val, dice_val)
    print(f"LR={lr}, BS={bs}: IoU={iou_val:.4f}, Dice={dice_val:.4f}")

# Random Search
print("\n Random Search")
param_space = list(product(
    [0.0001, 0.0005, 0.001, 0.002],
    ['Adam', 'RMSprop'],
    [0.0, 0.1, 0.2]
))
sampled_configs = random.sample(param_space, 5)
random_results = {}
for lr, opt, drop in sampled_configs:
    iou_val, dice_val = train_and_evaluate(dataset, lr=lr, optimizer_name=opt, dropout=drop)
    random_results[(lr, opt, drop)] = (iou_val, dice_val)
    print(f"Config: LR={lr}, Opt={opt}, Dropout={drop} → IoU={iou_val:.4f}, Dice={dice_val:.4f}")



 Manual Search
LR=0.001: IoU=0.4685, Dice=0.5844
LR=0.0005: IoU=0.4557, Dice=0.5833
LR=0.002: IoU=0.4316, Dice=0.5882

 Grid Search
LR=0.001, BS=1: IoU=0.4535, Dice=0.5768
LR=0.001, BS=2: IoU=0.5049, Dice=0.6701
LR=0.0005, BS=1: IoU=0.4986, Dice=0.6449
LR=0.0005, BS=2: IoU=0.2651, Dice=0.4044

 Random Search
Config: LR=0.0001, Opt=RMSprop, Dropout=0.0 → IoU=0.5167, Dice=0.6704
Config: LR=0.0005, Opt=RMSprop, Dropout=0.1 → IoU=0.3469, Dice=0.4898
Config: LR=0.002, Opt=RMSprop, Dropout=0.1 → IoU=0.4196, Dice=0.5752
Config: LR=0.001, Opt=RMSprop, Dropout=0.2 → IoU=0.4273, Dice=0.5568
Config: LR=0.0005, Opt=RMSprop, Dropout=0.0 → IoU=0.3364, Dice=0.4754


In [21]:
def summarize_results(results, label):
    best_config = max(results.items(), key=lambda x: x[1][0])  # Sort by IoU
    print(f"\n Best {label} Config → {best_config[0]}: IoU={best_config[1][0]:.4f}, Dice={best_config[1][1]:.4f}")

summarize_results(manual_results, "Manual Search")
summarize_results(grid_results, "Grid Search")
summarize_results(random_results, "Random Search")


 Best Manual Search Config → 0.001: IoU=0.4685, Dice=0.5844

 Best Grid Search Config → (0.001, 2): IoU=0.5049, Dice=0.6701

 Best Random Search Config → (0.0001, 'RMSprop', 0.0): IoU=0.5167, Dice=0.6704
